# 09. Hierarchical Clustering## What is this notebook about?Unlike K-Means (which gives flat groups), **Hierarchical Clustering** builds a **tree** of clusters from the bottom up:1. Start with each point as its own cluster.2. Repeatedly merge the two closest clusters.3. Stop when only one cluster remains.The full tree is called a **dendrogram**, and you can "cut" it at any level to get K flat clusters.## What you will learn1. How to compute hierarchical clustering with **scipy** and **sklearn**2. How to read a **dendrogram**3. How to **cut** the dendrogram to get K clusters4. How **linkage methods** (ward, complete, average, single) change the result## DatasetIris again - perfect for clustering since we know the true 3 species and can verify.

In [ ]:
# If you're running locally, uncomment and run this once.# In Google Colab, all of these are pre-installed - you can skip this cell.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebookimport numpy as np                 # numerical arraysimport pandas as pd                # tabular data (DataFrames)import matplotlib.pyplot as plt    # plottingimport seaborn as sns              # prettier statistical plots# Set up plot stylesns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)# Set a random seed so results are reproduciblenp.random.seed(42)

In [ ]:
from sklearn.datasets import load_irisfrom sklearn.cluster import AgglomerativeClusteringfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import adjusted_rand_scorefrom scipy.cluster.hierarchy import linkage, dendrogram, fcluster# Load and scale (always scale before distance-based methods)iris = load_iris(as_frame=True)X = StandardScaler().fit_transform(iris.data)y = iris.target.valuesprint(f"Shape: {X.shape}")

## Step 1. Build the linkage matrix`linkage()` runs the full hierarchical clustering and returns a matrix describing every merge.

In [ ]:
# Ward linkage: merges the pair of clusters that produces the smallest increase in within-cluster variance# It's the most popular defaultZ = linkage(X, method="ward")print(f"Linkage matrix shape: {Z.shape}")print("First 5 merges (each row = [cluster_a, cluster_b, distance, num_points_in_new_cluster]):")print(Z[:5])

## Step 2. Plot the dendrogramThis is the entire history of merges visualized.

In [ ]:
plt.figure(figsize=(14, 6))dendrogram(    Z,    color_threshold=10,    # color the branches below this distance    no_labels=True,         # too many points to label)plt.title("Iris Dendrogram (Ward Linkage)")plt.xlabel("Samples")plt.ylabel("Distance at which clusters merged")plt.axhline(y=10, color="red", linestyle="--", label="cut here for 3 clusters")plt.legend()plt.show()

**How to read this:**- Each leaf at the bottom is one flower.- Vertical lines going up show merges. Higher = clusters were further apart.- A horizontal line cutting across the tree creates clusters: each connected sub-tree below is one cluster.- Cut high → few big clusters. Cut low → many small clusters.The red dashed line shows where to cut for **3 clusters** - matching the 3 species!## Step 3. Cut the tree at K=3 and compare to true species

In [ ]:
# fcluster cuts the tree to give exactly 3 clusterslabels = fcluster(Z, t=3, criterion="maxclust")# Adjusted Rand Index measures how well the predicted clusters match the true labels# 1.0 = perfect match, 0 = randomari = adjusted_rand_score(y, labels)print(f"Adjusted Rand Index (vs true species): {ari:.3f}")print(f"  1.0 = perfect, 0.0 = random")# Visualize the clusters in 2Dplt.figure(figsize=(10, 5))sns.scatterplot(    x=iris.data["petal length (cm)"],    y=iris.data["petal width (cm)"],    hue=labels, palette="tab10", s=60,)plt.title("Hierarchical Clusters (cut at K=3) on Iris")plt.show()

ARI of ~0.6-0.8 typically. Iris has overlapping species, so perfect recovery is hard - but the algorithm got most of it right.## Step 4. Compare four linkage methodsLinkage = the rule for measuring distance between two clusters when deciding what to merge next.- **Single:** distance between the two closest points (creates "chains")- **Complete:** distance between the two farthest points (creates compact clusters)- **Average:** average pairwise distance- **Ward:** minimizes within-cluster variance (default, usually best)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))for ax, method in zip(axes.ravel(), ["single", "complete", "average", "ward"]):    Z = linkage(X, method=method)    dendrogram(Z, ax=ax, no_labels=True, color_threshold=0)    ax.set_title(f"Linkage: {method}")plt.tight_layout()plt.show()

Notice how dramatically different the trees look. **Ward** is usually a safe default.## Step 5. Exercises1. **Cut at different K values** (2, 3, 4, 5) and compute the ARI for each.2. **Try the Breast Cancer dataset** with hierarchical clustering. Does it separate malignant from benign?3. **Visualize a heatmap with clustered rows and columns** using `sns.clustermap`:   ```python   sns.clustermap(iris.data, cmap="viridis")   ```4. **Compare with K-Means** on the same data using ARI - which gives a higher score?5. **Try `sklearn.cluster.AgglomerativeClustering`** - it's the sklearn API for the same algorithm.Next: **10 - K-Nearest Neighbours**!